# Evaluate best model: XGBoost

In [ ]:
# imports
import pickle
import json
import glob
from pathlib import Path
import pandas as pd
import sys

# helpers
sys.path.insert(0, "../../")

from models_helper import ModelTrainer
from dataviz_helper import plot_confusion_matrices

In [ ]:
# define directories
DATA_DIR       = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"
MODELS_DIR     = "../../4_modeling/4_5_hyperparameter_tuning/models"
BEST_RUNS_PATH = "../5_1_get_best_runs_wandb/best_by_model_type.json"
OUTPUT_PATH    = "best_model_metrics.json"

# define label columns
LABEL_COLS = [
    "label_ifc_entity",
    "label_predefined_type",
    "label_is_external",
    "label_load_bearing",
]

# load feature names from the best run record
with open(BEST_RUNS_PATH, encoding="utf-8") as f:
    best_runs = json.load(f)

FEATURE_SUBSET = best_runs["xgboost"]["oversampling"]["best_val_f1_macro"]["feature_names"]
print(f"Features ({len(FEATURE_SUBSET)}): {FEATURE_SUBSET}")

Features (35): ['geom_layer_count', 'topo_unique_edge_count', 'mat_belag', 'mat_gips', 'topo_euler_characteristic', 'mat_kies', 'mat_beton', 'aabb_ratio_x_y', 'geom_ratio_area_vol', 'mat_zement', 'aabb_len_x', 'mat_metall', 'mat_bekleidung', 'mat_backstein', 'mat_kunststein', 'aabb_len_y', 'geom_compactness', 'aabb_len_z', 'aabb_max_z', 'aabb_ratio_z_y', 'tfbb_extent_2', 'mat_aluminium', 'tfbb_ratio_12', 'aabb_max_x', 'aabb_ratio_z_x', 'topo_connected_components', 'topo_avg_face_area', 'topo_face_count', 'mat_werkstoff', 'topo_vertex_edge_ratio', 'tfbb_extent_1', 'mat_allgemein', 'topo_vertex_count', 'tfbb_extent_0', 'aabb_min_z']


## 1. Load Data and Model

In [ ]:
# load validation and test datasets
df_val  = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

X_val  = df_val[FEATURE_SUBSET]
y_val  = df_val[LABEL_COLS]

print(f"Validation : {len(X_val):,} rows")

Validation : 8,458 rows
Test       : 3,549 rows


In [ ]:
# pick the best XGBoost model (highest score in filename and locally stored as pickle)
model_files = glob.glob(f"{MODELS_DIR}/best_xgboost_model_*.pkl")
model_path  = max(model_files, key=lambda p: float(Path(p).stem.split("_")[-1]))
print(f"Loading: {model_path}")

with open(model_path, "rb") as f:
    model: ModelTrainer = pickle.load(f)

print(f"Model type : {type(model).__name__}")
print(f"Labels     : {model.label_cols}")

Loading: ../../4_modeling/4_5_hyperparameter_tuning/models/best_xgboost_model_0.7276.pkl
Model type : ModelTrainer
Labels     : ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


## 2. Validation Set Evaluation

In [ ]:
# evaluate on validation set to check if everything is working as expected
val_metrics = model.evaluate(X_val, y_val)
val_metrics

,accuracy,precision,recall,mcc,f1_weighted,f1_macro
label,,,,,,
label_ifc_entity,0.8441,0.8427,0.8441,0.7749,0.8170,0.6748
label_predefined_type,0.5782,0.7824,0.5782,0.5381,0.6274,0.5357
label_is_external,0.8711,0.8719,0.8711,0.7910,0.8710,0.8939
label_load_bearing,0.7995,0.8171,0.7995,0.7101,0.7929,0.8061
mean,0.7732,0.8285,0.7732,0.7035,0.7771,0.7276


In [ ]:
# compute confusion matrices for all labels on the validation set
val_cms = model.confusion_matrices(X_val, y_val)

plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1)
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1)
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400)

In [ ]:
# compute confusion matrices for all labels on the validation set and show them normalized (by true label count)
plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400, normalize=True)